# 01 - Data Exploration: MovieLens 20M

This notebook performs exploratory data analysis on the MovieLens 20M dataset.
We examine rating distributions, user activity, movie popularity, sparsity,
temporal patterns, genre structure, and genome tags to inform modelling decisions.

**Dataset**: [MovieLens 20M](https://grouplens.org/datasets/movielens/20m/) by GroupLens Research

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from itertools import combinations

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

DATA_DIR = Path("../data/raw")
print(f"Data directory: {DATA_DIR.resolve()}")

## 1. Dataset Overview

Load each CSV file and inspect shapes, dtypes, and memory usage.

In [ ]:
# Load all datasets
ratings = pd.read_csv(DATA_DIR / "ratings.csv")
movies = pd.read_csv(DATA_DIR / "movies.csv")
links = pd.read_csv(DATA_DIR / "links.csv")
tags = pd.read_csv(DATA_DIR / "tags.csv")
genome_scores = pd.read_csv(DATA_DIR / "genome-scores.csv")
genome_tags = pd.read_csv(DATA_DIR / "genome-tags.csv")

datasets = {
    "ratings": ratings,
    "movies": movies,
    "links": links,
    "tags": tags,
    "genome_scores": genome_scores,
    "genome_tags": genome_tags,
}

print("=" * 70)
print("DATASET OVERVIEW")
print("=" * 70)
for name, df in datasets.items():
    mem_mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f"\n{name}:")
    print(f"  Shape: {df.shape}")
    print(f"  Memory: {mem_mb:.1f} MB")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Dtypes: {dict(df.dtypes)}")

In [ ]:
# Quick peek at each dataset
for name, df in datasets.items():
    print(f"\n--- {name} (first 3 rows) ---")
    display(df.head(3))

In [ ]:
# Missing values
print("Missing values per dataset:")
for name, df in datasets.items():
    missing = df.isnull().sum()
    if missing.any():
        print(f"\n{name}:")
        print(missing[missing > 0])
    else:
        print(f"  {name}: no missing values")

## 2. Rating Distribution

Analyse the distribution of rating values (0.5 to 5.0 in 0.5 increments).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of rating values
rating_counts = ratings["rating"].value_counts().sort_index()
axes[0].bar(rating_counts.index.astype(str), rating_counts.values, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Rating Values")
for i, (val, cnt) in enumerate(zip(rating_counts.index, rating_counts.values)):
    axes[0].text(i, cnt + cnt * 0.01, f"{cnt:,}", ha="center", fontsize=8)

# Summary statistics
stats_text = (
    f"Total ratings: {len(ratings):,}\n"
    f"Mean: {ratings['rating'].mean():.2f}\n"
    f"Median: {ratings['rating'].median():.1f}\n"
    f"Std: {ratings['rating'].std():.2f}\n"
    f"Unique users: {ratings['userId'].nunique():,}\n"
    f"Unique movies: {ratings['movieId'].nunique():,}"
)
axes[1].text(0.1, 0.5, stats_text, fontsize=14, family="monospace",
             transform=axes[1].transAxes, verticalalignment="center",
             bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))
axes[1].set_title("Summary Statistics")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 3. User Activity Distribution

How many ratings does each user provide? Understanding this helps set minimum thresholds.

In [ ]:
user_counts = ratings.groupby("userId").size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full distribution (log scale)
axes[0].hist(user_counts.values, bins=100, color="steelblue", edgecolor="white", log=True)
axes[0].axvline(user_counts.median(), color="red", linestyle="--", label=f"Median: {user_counts.median():.0f}")
axes[0].axvline(user_counts.mean(), color="orange", linestyle="--", label=f"Mean: {user_counts.mean():.0f}")
axes[0].set_xlabel("Number of Ratings per User")
axes[0].set_ylabel("Number of Users (log scale)")
axes[0].set_title("User Activity Distribution")
axes[0].legend()

# CDF
sorted_counts = np.sort(user_counts.values)
cdf = np.arange(1, len(sorted_counts) + 1) / len(sorted_counts)
axes[1].plot(sorted_counts, cdf, color="steelblue")
axes[1].axhline(0.5, color="gray", linestyle=":", alpha=0.5)
axes[1].axvline(20, color="red", linestyle="--", alpha=0.7, label="Min threshold (20)")
axes[1].set_xlabel("Number of Ratings per User")
axes[1].set_ylabel("Cumulative Proportion")
axes[1].set_title("CDF of User Activity")
axes[1].set_xscale("log")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"User activity percentiles:")
for p in [5, 25, 50, 75, 95, 99]:
    print(f"  {p}th percentile: {np.percentile(user_counts, p):.0f} ratings")

## 4. Movie Popularity Distribution

Examine how ratings are distributed across movies (long-tail distribution).

In [ ]:
movie_counts = ratings.groupby("movieId").size().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Log-log plot (power law)
axes[0].plot(range(1, len(movie_counts) + 1), movie_counts.values, color="steelblue", linewidth=0.5)
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].axhline(50, color="red", linestyle="--", alpha=0.7, label="Min threshold (50)")
axes[0].set_xlabel("Movie Rank (log)")
axes[0].set_ylabel("Number of Ratings (log)")
axes[0].set_title("Movie Popularity -- Long-Tail Distribution")
axes[0].legend()

# Histogram of ratings per movie
axes[1].hist(movie_counts.values, bins=100, color="steelblue", edgecolor="white", log=True)
axes[1].axvline(50, color="red", linestyle="--", label="Min threshold (50)")
axes[1].set_xlabel("Number of Ratings per Movie")
axes[1].set_ylabel("Number of Movies (log scale)")
axes[1].set_title("Movie Popularity Histogram")
axes[1].legend()

plt.tight_layout()
plt.show()

# Top 10 most-rated movies
top_movies = movie_counts.head(10).reset_index()
top_movies.columns = ["movieId", "n_ratings"]
top_movies = top_movies.merge(movies[["movieId", "title"]], on="movieId")
print("\nTop 10 Most-Rated Movies:")
for i, row in top_movies.iterrows():
    print(f"  {i+1}. {row['title']} ({row['n_ratings']:,} ratings)")

## 5. Sparsity Analysis

How sparse is the user-item matrix? This drives the choice of models.

In [ ]:
n_users = ratings["userId"].nunique()
n_movies = ratings["movieId"].nunique()
n_ratings = len(ratings)
n_possible = n_users * n_movies
sparsity = 1.0 - (n_ratings / n_possible)
density = n_ratings / n_possible

print("User-Item Matrix Statistics")
print("=" * 40)
print(f"Users:          {n_users:>10,}")
print(f"Movies:         {n_movies:>10,}")
print(f"Ratings:        {n_ratings:>10,}")
print(f"Possible:       {n_possible:>10,}")
print(f"Sparsity:       {sparsity:>10.4%}")
print(f"Density:        {density:>10.6%}")
print(f"\nThe matrix is {sparsity:.2%} empty.")
print(f"On average, each user has rated {n_ratings/n_users:.0f} of {n_movies:,} movies.")

## 6. Temporal Analysis

Ratings over time reveal trends, seasonal patterns, and potential temporal splits.

In [ ]:
ratings["datetime"] = pd.to_datetime(ratings["timestamp"], unit="s")
ratings["year"] = ratings["datetime"].dt.year
ratings["month"] = ratings["datetime"].dt.to_period("M")

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Ratings per year
yearly = ratings.groupby("year").size()
axes[0].bar(yearly.index, yearly.values, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Number of Ratings")
axes[0].set_title("Ratings per Year")

# Monthly trend
monthly = ratings.groupby("month").size()
axes[1].plot(monthly.index.to_timestamp(), monthly.values, color="steelblue", linewidth=0.8)
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Number of Ratings")
axes[1].set_title("Monthly Rating Volume")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

print(f"Date range: {ratings['datetime'].min()} to {ratings['datetime'].max()}")
print(f"Time span: {(ratings['datetime'].max() - ratings['datetime'].min()).days / 365:.1f} years")

In [ ]:
# Average rating over time (quality drift)
yearly_avg = ratings.groupby("year")["rating"].mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(yearly_avg.index, yearly_avg.values, marker="o", color="steelblue")
ax.axhline(ratings["rating"].mean(), color="red", linestyle="--", alpha=0.5, label=f"Overall mean: {ratings['rating'].mean():.2f}")
ax.set_xlabel("Year")
ax.set_ylabel("Average Rating")
ax.set_title("Average Rating by Year (Rating Inflation / Deflation)")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Genre Analysis

Movies can belong to multiple genres. Analyse genre frequency and co-occurrence.

In [ ]:
# Explode genres
movies["genre_list"] = movies["genres"].str.split("|")
genre_exploded = movies.explode("genre_list")

# Genre frequency
genre_counts = genre_exploded["genre_list"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
genre_counts.plot(kind="barh", ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_xlabel("Number of Movies")
axes[0].set_ylabel("Genre")
axes[0].set_title("Genre Frequency")
axes[0].invert_yaxis()

# Number of genres per movie
genres_per_movie = movies["genre_list"].apply(len)
axes[1].hist(genres_per_movie, bins=range(1, genres_per_movie.max() + 2), color="steelblue",
             edgecolor="white", align="left")
axes[1].set_xlabel("Number of Genres")
axes[1].set_ylabel("Number of Movies")
axes[1].set_title("Genres per Movie")

plt.tight_layout()
plt.show()

print(f"Total unique genres: {genre_counts.shape[0]}")
print(f"Average genres per movie: {genres_per_movie.mean():.1f}")

In [ ]:
# Genre co-occurrence matrix
all_genres = sorted(genre_counts.index.tolist())
# Remove "(no genres listed)" if present
all_genres = [g for g in all_genres if g != "(no genres listed)"]

cooccurrence = pd.DataFrame(0, index=all_genres, columns=all_genres, dtype=int)
for genre_list in movies["genre_list"]:
    valid = [g for g in genre_list if g in all_genres]
    for g1, g2 in combinations(valid, 2):
        cooccurrence.loc[g1, g2] += 1
        cooccurrence.loc[g2, g1] += 1
    for g in valid:
        cooccurrence.loc[g, g] += 1

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(cooccurrence, dtype=bool), k=1)
sns.heatmap(cooccurrence, mask=mask, annot=False, fmt="d", cmap="YlOrRd",
            ax=ax, square=True, linewidths=0.5)
ax.set_title("Genre Co-occurrence Matrix")
plt.tight_layout()
plt.show()

## 8. Genome Tag Analysis

The genome scores capture how strongly a tag applies to each movie (0-1 relevance).

In [ ]:
print(f"Genome scores shape: {genome_scores.shape}")
print(f"Unique movies with genome: {genome_scores['movieId'].nunique()}")
print(f"Unique tags: {genome_scores['tagId'].nunique()}")
print(f"\nGenome tags sample:")
display(genome_tags.head(10))

# Average relevance per tag across all movies
tag_avg_relevance = genome_scores.groupby("tagId")["relevance"].mean().reset_index()
tag_avg_relevance = tag_avg_relevance.merge(genome_tags, on="tagId")
tag_avg_relevance = tag_avg_relevance.sort_values("relevance", ascending=False)

print("\nTop 20 most relevant genome tags (avg across all movies):")
for i, row in tag_avg_relevance.head(20).iterrows():
    print(f"  {row['tag']:30s}  relevance: {row['relevance']:.3f}")

In [ ]:
# Distribution of genome relevance scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sample for histogram (full dataset is huge)
sample = genome_scores["relevance"].sample(n=500_000, random_state=42)
axes[0].hist(sample, bins=50, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Relevance Score")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Distribution of Genome Relevance Scores (sample)")

# Top 20 tags bar chart
top_tags = tag_avg_relevance.head(20)
axes[1].barh(top_tags["tag"], top_tags["relevance"], color="steelblue", edgecolor="white")
axes[1].set_xlabel("Average Relevance")
axes[1].set_title("Top 20 Genome Tags by Average Relevance")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 9. Key Findings Summary

| Finding | Value | Implication |
|---------|-------|-------------|
| Total ratings | ~20M | Large enough for deep collaborative filtering |
| Unique users | ~138K | Reasonable user base for CF |
| Unique movies | ~27K | Manageable item catalog |
| Sparsity | >99% | Sparse matrix techniques essential; ALS well-suited |
| Rating distribution | Left-skewed (mode at 4.0) | Users tend to rate movies they like |
| Long-tail movies | Many movies with <50 ratings | Content-based helps for cold items |
| Genome tags | 1,128 tags per movie | Rich content signal for content-based models |
| Time span | ~20 years | Temporal split is appropriate for evaluation |

### Modelling implications

1. **Collaborative filtering** is viable given 20M interactions, but extreme sparsity means item-item CF should use top-k truncated similarities.
2. **Content-based** with genome tags can address the cold-start problem for long-tail movies.
3. **ALS matrix factorisation** is well-suited to the sparse, large-scale user-item matrix.
4. **Hybrid blending** of CF, content, and ALS should outperform any single approach.
5. **Minimum thresholds** of 20 ratings/user and 50 ratings/movie will filter noise while retaining most interactions.

In [ ]:
# Clean up temporary columns
ratings.drop(columns=["datetime", "year", "month"], inplace=True, errors="ignore")
movies.drop(columns=["genre_list"], inplace=True, errors="ignore")

print("Exploration complete.")